# Lower Ohio River Flood Valley 4/3/25 

## Stations:
## - Indiana: GALENA 4.3 ENE, CANNELTON 6.8 NE, HENRYVILLE 2.6 E 
## - Kentucky: GREENVILLE 6.2 WSW, EDDYVILLE 2.0 NNE, 

In [8]:
from datetime import datetime, timedelta
import numpy as np
import torch
from earth2studio.data import GEFS_FX, HRRR, GEFS_FX_721x1440
import io
import tarfile
from pathlib import Path
import numpy as np
import requests
import tqdm
import matplotlib.pyplot as plt


GEFS_SELECT_VARIABLES = [
    "u10m",
    "v10m",
    "t2m",
    "r2m",
    "sp",
    "msl",
    "tcwv",
]

GEFS_VARIABLES = [
    "u1000",
    "u925",
    "u850",
    "u700",
    "u500",
    "u250",
    "v1000",
    "v925",
    "v850",
    "v700",
    "v500",
    "v250",
    "z1000",
    "z925",
    "z850",
    "z700",
    "z500",
    "z200",
    "t1000",
    "t925",
    "t850",
    "t700",
    "t500",
    "t100",
    "r1000",
    "r925",
    "r850",
    "r700",
    "r500",
    "r100",
]

ds_gefs = GEFS_FX(cache=True)
ds_gefs_select = GEFS_FX_721x1440(cache=True, member="gec00")


def fetch_input_gefs(
    time: datetime, lead_time: timedelta, content_dtype: str = "float32"
):
    """Fetch input GEFS data and place into a single numpy array."""
    dtype = np.dtype(getattr(np, content_dtype))
    g = np.array(9.80665, dtype=dtype)  # standard gravity (m/s^2)

    # Fetch high-res select GEFS input data
    select_data = ds_gefs_select(time, lead_time, GEFS_SELECT_VARIABLES).values
    # Crop to bounding box [225, 21, 300, 53]
    select_data = select_data[:, 0, :, 148:277, 900:1201].astype(dtype)
    assert select_data.shape == (1, len(GEFS_SELECT_VARIABLES), 129, 301)

    # Fetch GEFS pressure-level input data
    pressure_data = ds_gefs(time, lead_time, GEFS_VARIABLES)

    # Interpolate to 0.25° grid (keeping your existing approach)
    pressure_data = torch.nn.functional.interpolate(
        torch.as_tensor(pressure_data.values),
        (len(GEFS_VARIABLES), 721, 1440),
        mode="nearest",
    ).numpy()

    # Crop to bounding box [225, 21, 300, 53]
    pressure_data = pressure_data[:, 0, :, 148:277, 900:1201].astype(dtype)
    assert pressure_data.shape == (1, len(GEFS_VARIABLES), 129, 301)

    # Divide all z-levels by g to convert geopotential (m^2/s^2) -> geopotential height (m)
    z_vars = {"z1000", "z925", "z850", "z700", "z500", "z200"}
    z_indices = [i for i, v in enumerate(GEFS_VARIABLES) if v in z_vars]
    pressure_data[:, z_indices, :, :] /= g

    # Create lead time field in 3-hour increments
    lead_hour = (int(lead_time.total_seconds() // (3 * 60 * 60)) * np.ones((1, 1, 129, 301), dtype=dtype))

    input_data = np.concatenate([select_data, pressure_data, lead_hour], axis=1)[None]
    return input_data

In [15]:
import cartopy.crs as ccrs
import matplotlib.colors as colors
import matplotlib.cm as cm
import matplotlib as mpl
import cartopy.feature as cfeature

lats = np.load("corrdiff_output_lat.npy")
lons = np.load("corrdiff_output_lon.npy")
lons = lons - 360


# Find index for Ohio river 
# GALENA 4.3 ENE
Galena_lat = np.where((-85.95 < lons) & (lons < -85.8) & (lats > 38.383) & (lats < 38.385))

#CANNELTON 6.8 NE
Cannelton_lat = np.where((-86.645 < lons) & (lons < -86.64) & (lats > 37.9) &(lats < 38))
#print(Cannelton_lat)

#HENRYVILLE 2.6 E
Henryville_lat = np.where((-85.72 < lons) & (lons < -85.7) & (lats > 38.52) &(lats < 38.54))
#print(Henryville_lat)

#GREENVILLE 6.2 WSW
Greenville_lat = np.where((-87.3 < lons) & (lons < -87.27) & (lats > 37.15) &(lats < 37.158))
#print(Greenville_lat)

#EDDYVILLE 2.0 NNE
Eddyville_lat = np.where((-88.07 < lons) & (lons < -88.05) & (lats > 37.1) &(lats < 37.15))
#print(Eddyville_lat)

#GRAND RIVERS 3.9 S
GrandRiver_lat = np.where((-88.234 < lons) & (lons < -88.231) & (lats > 36.99) &(lats < 37.05))
#print(GrandRiver_lat)

#Lewisburg 0.2 E
Lewisburg_lat = np.where((-86.95 < lons) & (lons < -86.9) & (lats > 36.98) & (lats < 36.99))
#print(Lewisburg_lat)


(array([491]), array([1174]))


In [25]:

year = 2025
month = 4
day = 3
Hours = [24]

#Total_precp_Gal = []

#Total_precp_Can = []

#Total_precp_Her = []

#Total_precp_Lew = []

#Total_precp_Gre = []

#Total_precp_Edd = []

#Total_precp_Gra = []


for H in Hours:
    time = datetime(year, month, day) # initial condition 
    lead_time1 = timedelta(hours = H) # model run time, note that corrdiff only allows 24hr prediction
    input_array = fetch_input_gefs(time, lead_time1)

    np.save("corrdiff_inputs_ken.npy", input_array)

    url = f"http://corrdiff-nim-service-laurahu:8000/v1/infer"
    files = {
        "input_array": ("input_array", open("corrdiff_inputs_ken.npy", "rb")),
    }
    data = {
        "samples": 5, # users can choose how many samples they want to generate 
        "steps": 10, # users can also choose how many steps they want the diffusion mod to take 
        "seed": 0,
    }
    headers = {
        "accept": "application/x-tar",
    }
    print("Sending post request to NIM")
    r = requests.post(url, headers=headers, data=data, files=files, timeout=3000)
    if r.status_code != 200:
        raise Exception(r.content)
    else:
    
        with open("output.tar", "wb") as tar:
            tar.write(r.content)
    print("Done!")
    
    input = np.load("corrdiff_inputs_ken.npy")
    
    prec_mean = []
    
    with tarfile.open("output.tar") as tar:
        for i, member in enumerate(tar.getmembers()):
            arr_file = io.BytesIO()
            arr_file.write(tar.extractfile(member).read())
            arr_file.seek(0)
            data = np.load(arr_file)
 
            prec = data[0, 0, 3]

            prec_mean.append(prec)
    
    
    #ensemble_mean
    precp = np.mean(prec_mean, axis = 0)

    Total_precp_Gal.append(precp[545, 1231])

    Total_precp_Can.append(precp[525, 1212])

    Total_precp_Her.append(precp[551, 1236])

    Total_precp_Lew.append(precp[490, 1207])

    Total_precp_Gre.append(precp[495, 1196])

    Total_precp_Edd.append(precp[491, 1174])

    Total_precp_Gra.append(precp[488, 1169])

    

    

    


Fetching GEFS data: 100%|██████████| 7/7 [00:00<00:00, 40.71it/s]


2026-04-15 19:35:32.532 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2sp25/gec00.t00z.pgrb2s.0p25.f024 1763565-742388
2026-04-15 19:35:32.557 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2sp25/gec00.t00z.pgrb2s.0p25.f024 4003206-423725
2026-04-15 19:35:32.582 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2sp25/gec00.t00z.pgrb2s.0p25.f024 6363946-832227
2026-04-15 19:35:32.607 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2sp25/gec00.t00z.pgrb2s.0p25.f024 7196173-814554
2026-04-15 19:35:32.631 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2sp25/gec00.t00z.pgrb2s.0p25.f024 4865518-656960
2026-04-15 19:35:32.655 | DEBUG    

Fetching GEFS data:   0%|          | 0/30 [00:00<?, ?it/s]

2026-04-15 19:35:32.708 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 2624060-208523
2026-04-15 19:35:32.721 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 5789481-161177


Fetching GEFS data:   0%|          | 0/30 [00:00<?, ?it/s]

2026-04-15 19:35:32.733 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 7240461-253269
2026-04-15 19:35:32.745 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 5439568-116742
2026-04-15 19:35:32.756 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 6978776-261685
2026-04-15 19:35:32.768 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 6116818-245463
2026-04-15 19:35:32.779 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 6720840-257936
2026-04-15 19:35:32.790 | DEBUG    | ear

Fetching GEFS data:   0%|          | 0/30 [00:00<?, ?it/s]

2026-04-15 19:35:32.913 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 5204058-235510
2026-04-15 19:35:32.924 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 1907111-109880


Fetching GEFS data: 100%|██████████| 30/30 [00:00<00:00, 89.50it/s]


2026-04-15 19:35:32.934 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 10243810-265480
2026-04-15 19:35:32.946 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 4150749-164376
2026-04-15 19:35:32.957 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 6362281-123611
2026-04-15 19:35:32.967 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 9352138-264884
2026-04-15 19:35:32.978 | DEBUG    | earth2studio.data.gefs:fetch_array:405 - Fetching GEFS grib file: noaa-gefs-pds/gefs.20250403/00/atmos/pgrb2ap5/gec00.t00z.pgrb2a.0p50.f024 10509290-265706
2026-04-15 19:35:32.989 | DEBUG    | e

In [26]:
len(Total_precp_Gra)

8

In [27]:
import pandas as pd

df_all = pd.DataFrame({
    # Cincinnati
    'precp_Gal': Total_precp_Gal,
    
    # Shawneetown
    'precp_Can': Total_precp_Can,

    # Olmsted Dam
    'precp_Her': Total_precp_Her,

    # Newburgh
    'precp_Lew': Total_precp_Lew,

    # Owensboro
    'precp_Gre': Total_precp_Gre,

    # Louisville
    'precp_Edd': Total_precp_Edd,

    # Madison
    'precp_Gra': Total_precp_Gra
})

# Save to CSV so it persists after kernel restart
df_all.to_csv('total_precipitation_Lower_data.csv', index=False)
print('Saved! Shape:', df_all.shape)
display(df_all.head())

Saved! Shape: (8, 7)


,precp_Gal,precp_Can,precp_Her,precp_Lew,precp_Gre,precp_Edd,precp_Gra
0,0.009334,0.017363,0.005043,0.039080,0.034647,3.077855,2.373804
1,11.468384,12.602197,9.380953,0.148197,0.688574,18.438374,18.172773
2,0.821825,1.616156,0.627998,3.691392,1.685926,3.025757,2.877212
3,0.041964,0.369966,0.039032,2.097384,0.916567,1.837361,1.693880
4,0.027715,0.236451,0.010783,1.337115,0.544175,1.194281,1.031724


References: https://www.weather.gov/pah/April2025Flooding#ilrain